# Experimental Guard in NRTK

This notebook demonstrates how the experimental guard works in NRTK.

The perturbers used in this notebook are subject to change. The main purpose of this notebook is to showcase the experimental guard, not to show how the experimental perturbers are used.

## Set Up the Environment

In [1]:
%pip install -qU pip
print("Installing nrtk with required extras...")
%pip install -q "nrtk[pillow,pyav]"
print("Installing notebook-specific packages...")
%pip install -q matplotlib
print("Done!")

from nrtk.utils._extras import print_extras_status  # noqa: E402 - intentionally after %pip install

print_extras_status()

Note: you may need to restart the kernel to use updated packages.
Installing nrtk with required extras...


Note: you may need to restart the kernel to use updated packages.
Installing notebook-specific packages...


Note: you may need to restart the kernel to use updated packages.
Done!


Detected status of NRTK extras and their dependencies:



[albumentations]
  - nrtk-albumentations       ✗ missing

[diffusion]
  - torch                     ✗ missing
  - diffusers                 ✗ missing
  - accelerate                ✗ missing
  - Pillow                    ✓ 12.3.0
  - transformers              ✗ missing
  - protobuf                  ✗ missing

[graphics]
  - opencv-python             ✗ missing

[hcipy]
  - hcipy                     ✗ missing
  - scipy                     ✗ missing
  - xxhash                    ✗ missing

[headless]
  - opencv-python-headless    ✗ missing

[maite]
  - maite                     ✗ missing

[pillow]
  - Pillow                    ✓ 12.3.0

[pyav]
  - av                        ✓ 17.1.0

[pybsm]
  - pybsm                     ✗ missing

[skimage]
  - scikit-image              ✗ missing

[tools]
  - kwcoco                    ✗ missing
  - Pillow                    ✓ 12.3.0
  - click                     ✓ 8.5.0
  - fastapi                   ✗ missing
  - uvicorn                   ✗ missing
  - pyd

    https://nrtk.readthedocs.io/en/stable/



We will use a sample video of a highway to demonstrate the experimental import guard. The cell below will download a dataset containing only the sample video.

In [2]:
import glob
import os
import sys
import urllib.request
import zipfile
from pathlib import Path

data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)
HMIE_PATH = os.path.join(data_dir, "highway_hmie")
if not os.path.isdir(HMIE_PATH):
    # The Girder folder download streams a zip whose root folder is
    # highway_hmie/, so extracting into data_dir creates HMIE_PATH.
    url = "https://data.kitware.com/api/v1/folder/6a67b8461866d4fa1981b9be/download"
    zip_path = os.path.join(data_dir, "highway_hmie.zip")
    _ = urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_dir)
    os.remove(zip_path)

video_path = glob.glob(os.path.join(HMIE_PATH, "**", "seq_*", "*.mp4"), recursive=True)[0]
print(f"Using video from HMIE dataset: {video_path}")

Using video from HMIE dataset: ./data/highway_hmie/out_000000_000000/out_000000_000001/seq_mp4/out_000000_000001.mp4


## Experimental Guard

Experimental features are subject to sudden changes without advance notice. As a result, we place these features behind an experimental guard.  

If a user tries to import an experimental perturber, an ``ImportError`` will occur. The error will mention the feature is experimental and explain how to enable the experimental feature. Once a feature moves out of experimental, it can be imported as normal and will no longer be subject to changes without warning.

Below is an example showcasing the error raised by an experimental perturber (``FramewisePerturber``).

In [3]:
try:
    from nrtk.impls.perturb_video import FramewisePerturber
except ImportError as e:
    print(f"ImportError: {e}")
else:
    raise AssertionError("Importing FramewisePerturber succeeded without `import nrtk.experimental`")

ImportError: nrtk.impls.perturb_video.FramewisePerturber is experimental. Enable it with `import nrtk.experimental` before importing it. Experimental APIs may change without a deprecation warning.


To use experimental features, a user will need to include ``import nrtk.experimental`` before attempting to import the experimental feature. Once this line is included, a warning noting experimental features are now enabled will be emitted. Experimental features will then function like any other NRTK feature.

Below is an example that disables the import guard and then displays the perturbed video next to the original video.

In [4]:
import nrtk.experimental  # noqa: F401
from nrtk.impls.perturb_image.photometric.enhance import BrightnessPerturber
from nrtk.impls.perturb_video import FramewisePerturber
from nrtk.interfaces import VideoFrame

# notebook_utils lives in docs/examples
for _anc in [Path.cwd(), *Path.cwd().parents]:
    if (_anc / "docs" / "examples" / "notebook_utils").is_dir():
        sys.path.insert(0, str(_anc / "docs" / "examples"))
        break
else:
    raise ImportError("Cannot locate notebook_utils")

from notebook_utils.video import (  # pyright: ignore[reportMissingImports] # noqa: E402 - intentionally after finding notebook_utils
    read_video,
    video_comparison,
)

video_frames = read_video(video_path, max_frames=120)
images = [frame.image for frame in video_frames]
dt_ms = int(1000 * (video_frames[1].timestamp - video_frames[0].timestamp)) if len(video_frames) > 1 else 42

perturber = FramewisePerturber(frame_perturber=BrightnessPerturber(factor=1.5))
frames = (VideoFrame(image=frame.image, timestamp=frame.timestamp) for frame in video_frames)
perturbed_images = [frame.image for frame in perturber(frames=frames)]

video_comparison(
    frame_sequences=[images, perturbed_images],
    titles=["Original", "Brightness factor 1.5"],
    interval=dt_ms,
)

/tmp/ipykernel_16142/1803298625.py:1: ExperimentalWarning: Experimental NRTK features are now enabled. Their APIs may change without a deprecation warning.
  import nrtk.experimental  # noqa: F401
